# Inference Failover

Deploy two model-safe AI Gateway routes that exercise priority and weighted fallback across regional Azure OpenAI deployments while collecting focused LLM telemetry. See [README.md](README.md) for deployment details and prerequisites.

## What This Sample Does

- Deploys nine regional `Standard` Azure OpenAI deployments at `1` thousand TPM for deliberate pressure testing.
- Exposes only `POST /inference/gpt-5-1/chat/completions` and `POST /inference/gpt-4-1-mini/chat/completions`.
- Routes each model through its own APIM backend pool, with managed identity authentication, TLS validation, circuit breaking, and buffered retries.
- Captures AOAI request, response, latency, and token telemetry in an Azure Monitor Workbook and in the verification charts below.

## Route Graph

Each operation targets one APIM backend pool. The pool selects a single member per request by priority (lowest first), and members that share a priority split traffic by weight. The backends never call one another; APIM picks the active backend and fails over to the next priority tier when one is unavailable.

Every target is a regional `Standard` pay-as-you-go deployment. `PTU` identifies a simulated routing preference, not a provisioned deployment.

```mermaid
flowchart LR
  subgraph S5 [" gpt-5.1 (East US 2)"]
    C51["POST /inference/gpt-5-1/chat/completions"] --> P5{{"&nbsp;&nbsp;Backend pool: inference-gpt-5-1-pool&nbsp;&nbsp;"}}
    P5 -- "priority 1" --> A["`East US 2
                            PTU
                            gpt-5-1`"]
    P5 -- "priority 2" --> D["`West US 3
                            PTU
                            gpt-5-1`"]
    P5 -- "priority 3" --> B["`East US 2
                            PAYG
                            gpt-5-1`"]
    P5 -- "priority 4 &middot; weight 50" --> E["`West US 3 
                                                PAYG
                                                gpt-5-1`"]
    P5 -- "priority 4 &middot; weight 50" --> G["`South Central US
                                                PAYG
                                                gpt-5-1`"]
  end

  S5 ~~~ S4

  subgraph S4 [" gpt-4.1-mini (East US 2)"]
    C41["POST /inference/gpt-4-1-mini/chat/completions"] --> P4{{"&nbsp;&nbsp;Backend pool: inference-gpt-4-1-mini-pool&nbsp;&nbsp;"}}
    P4 -- "priority 1" --> C["`East US 2
                                PTU
                                gpt-4-1-mini`"]
    P4 -- "priority 2" --> F["`West US 3 
                                PTU
                                gpt-4-1-mini`"]
    P4 -- "priority 3" --> DP["`East US 2
                                PAYG
                                gpt-4-1-mini`"]
    P4 -- "priority 4" --> H["`South Central US
                                PAYG
                                gpt-4-1-mini`"]
  end
```


In [ ]:
import azure_resources as az
import utils
from apimtypes import API, APIM_SKU, APIOperation, HTTP_VERB, INFRASTRUCTURE, Region
from console import print_ok

# ------------------------------
#    USER CONFIGURATION
# ------------------------------

rg_location = Region.EAST_US_2
index = 1
apim_sku = APIM_SKU.BASICV2
deployment = INFRASTRUCTURE.SIMPLE_APIM
api_prefix = 'inference'
tags = ['inference-failover', 'ai-gateway', 'observability']
max_completion_tokens = 128                 # Per-request completion cap for the pressure scenarios

# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------

sample_folder = 'inference-failover'
rg_name = az.get_infra_rg_name(deployment, index)
supported_infras = [
    INFRASTRUCTURE.AFD_APIM_PE,
    INFRASTRUCTURE.APIM_ACA,
    INFRASTRUCTURE.APPGW_APIM,
    INFRASTRUCTURE.APPGW_APIM_PE,
    INFRASTRUCTURE.SIMPLE_APIM,
]
nb_helper = utils.NotebookHelper(
    sample_folder,
    rg_name,
    rg_location,
    deployment,
    supported_infras,
    index=index,
    apim_sku=apim_sku,
)
current_user, current_user_id, tenant_id, subscription_id = az.get_account_info()

policy_template = utils.read_policy_xml('inference-api-policy.xml', sample_name=sample_folder)

def build_inference_policy(backend_pool_id: str, retry_count: int) -> str:
    """Build one model-safe inference API policy from the shared template."""
    return policy_template.replace('BACKEND_POOL_ID', backend_pool_id).replace('RETRY_COUNT', str(retry_count))

chat_operation = APIOperation(
    'chat-completions',
    'Chat Completions',
    '/chat/completions',
    HTTP_VERB.POST,
    'Send an inference request through a model-safe backend pool.',
)
gpt_5_1_api = API(
    'inference-gpt-5-1',
    'Inference gpt-5.1',
    f'{api_prefix}/gpt-5-1',
    'Priority and weighted failover for gpt-5.1 deployments.',
    policyXml=build_inference_policy('inference-gpt-5-1-pool', 4),
    operations=[chat_operation],
    tags=tags,
    enableLlmLogging=True,
)
gpt_4_1_mini_api = API(
    'inference-gpt-4-1-mini',
    'Inference gpt-4.1-mini',
    f'{api_prefix}/gpt-4-1-mini',
    'Priority failover for gpt-4.1-mini deployments.',
    policyXml=build_inference_policy('inference-gpt-4-1-mini-pool', 3),
    operations=[chat_operation],
    tags=tags,
    enableLlmLogging=True,
)
apis = [gpt_5_1_api, gpt_4_1_mini_api]

print_ok('Notebook initialized')


## Deploy Inference Gateway Resources

Deploy the regional Azure OpenAI constellation, APIM backends and pools, retry fragment, LLM diagnostic wiring, two APIs, and the AOAI-only workbook into the selected infrastructure resource group.

In [ ]:
from console import print_error, print_ok, print_val

bicep_parameters = {
    'location': {'value': rg_location},
    'index': {'value': index},
    'apis': {'value': [api.to_dict() for api in apis]},
}
output = nb_helper.deploy_sample(bicep_parameters)

if output.success:
    apim_name = output.get('apimServiceName', 'APIM Service Name')
    apim_gateway_url = output.get('apimResourceGatewayURL', 'APIM API Gateway URL')
    log_analytics_name = output.get('logAnalyticsWorkspaceName', 'Log Analytics Workspace Name')
    log_analytics_id = output.get('logAnalyticsWorkspaceId', 'Log Analytics Workspace ID')
    workbook_name = output.get('workbookName', 'Workbook Name')
    workbook_id = output.get('workbookId', 'Workbook ID')
    api_outputs = output.getJson('apiOutputs', 'Inference API outputs', secure=True)
    api_keys = {api_output['name']: api_output['subscriptionPrimaryKey'] for api_output in api_outputs}
    print_val('Workbook', workbook_name)
    print_ok('Inference failover deployment completed successfully')
else:
    print_error('Deployment failed')
    raise SystemExit(1)

## 🧪 Inference Failover Test Matrix

Run a methodical series of scenarios that exercise the model-safe backend pools under increasing pressure, mirroring the structure of the load-balancing sample. Each scenario sends several real Azure OpenAI requests through a single shared session, captures the served backend from the `X-Backend-Id` response header, and verifies the observed behaviour.

| # | Scenario | API | Runs | Sleep | Behaviour verified |
| --- | --- | --- | --- | --- | --- |
| 1 | Baseline warm path | both | 6 each | none | Fresh priority-1 backend serves small requests with `200` |
| 2 | Sustained pressure | gpt-5.1 | 45 | none | Low-TPM exhaustion forces throttling and priority failover |
| 3 | Sustained pressure | gpt-4.1-mini | 30 | none | Independent pool exhausts and fails over on its own |
| 4 | Paced recovery | gpt-5.1 | 30 | 1.5 s | Inter-request spacing lets circuit breakers recover and lifts the success rate |
| 5 | Terminal exhaustion | gpt-5.1 | 35 | none | A hard burst drains every tier and returns the generic `503` |

> These are real Azure OpenAI calls (~150 requests) that consume tokens. The deployments are intentionally provisioned at minimum capacity so that pressure is observable quickly; this is an observation aid, not production sizing guidance.


In [ ]:
import json
import time

import requests as http_requests

from apimtesting import ApimTesting
from console import print_error, print_info, print_message, print_ok

if 'api_keys' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

endpoint_url, request_headers, allow_insecure_tls = utils.get_endpoint(deployment, rg_name, apim_gateway_url)

# Stable backend -> chart index mapping in pool-priority order so chart colors track failover tiers.
gpt_5_1_backend_index = {
    'a-gpt-5-1': 0,   # P1 in-region PTU
    'b-gpt-5-1': 1,   # P2 in-region PAYG
    'd-gpt-5-1': 2,   # P3 out-of-region PTU
    'e-gpt-5-1': 3,   # P4 out-of-region PAYG (westus3)
    'g-gpt-5-1': 4,   # P4 out-of-region PAYG (southcentralus)
}
gpt_4_1_mini_backend_index = {
    'c-gpt-4-1-mini': 0,   # P1 in-region PTU
    'd-gpt-4-1-mini': 1,   # P2 in-region PAYG
    'f-gpt-4-1-mini': 2,   # P3 out-of-region PTU
    'h-gpt-4-1-mini': 3,   # P4 out-of-region PAYG
}

gpt_5_1_route = f'{api_prefix}/gpt-5-1/chat/completions'
gpt_4_1_mini_route = f'{api_prefix}/gpt-4-1-mini/chat/completions'

small_payload = {
    'messages': [{'role': 'user', 'content': 'Return the word ready.'}],
    'max_completion_tokens': 16,
}
pressure_payload = {
    'messages': [{'role': 'user', 'content': 'Summarize failover readiness in three bullets. ' * 20}],
    'max_completion_tokens': max_completion_tokens,
}


def run_inference_scenario(session, route, api_key, payload, runs, backend_index_map, scenario_label, sleep_ms = 0):
    """Send `runs` POSTs through one route, attributing each response to its served backend."""
    results = []
    url = f'{endpoint_url}/{route}'
    print_message(scenario_label, blank_above = True)
    print_info(f'POST {url}')

    for run_index in range(1, runs + 1):
        start_time = time.time()
        response = session.post(url, headers = {'api-key': api_key}, json = payload, timeout = 120)
        response_time = time.time() - start_time

        backend_id = response.headers.get('X-Backend-Id', 'unknown')
        backend_index = backend_index_map.get(backend_id, 99)

        # Inject the backend index into the stored body so charts.BarChart colors each 200 bar by the
        # served backend. Non-200 responses keep their error body and are charted in red.
        if response.status_code == 200:
            response_body = json.dumps({'index': backend_index, 'backend': backend_id})
        else:
            response_body = response.text

        results.append({
            'run': run_index,
            'response': response_body,
            'status_code': response.status_code,
            'response_time': response_time,
            'headers': dict(response.headers),
            'backend': backend_id,
        })

        print_info(f'Run {run_index}/{runs}: {response.status_code} via {backend_id} ({response_time:.2f}s)')

        if sleep_ms:
            time.sleep(sleep_ms / 1000)

    return results


def summarize_scenario(results):
    """Return (successes, throttled, unavailable, served_backends) for a scenario result list."""
    successes = sum(1 for r in results if r['status_code'] == 200)
    throttled = sum(1 for r in results if r['status_code'] == 429)
    unavailable = sum(1 for r in results if r['status_code'] == 503)
    served_backends = sorted({r['backend'] for r in results if r['status_code'] == 200 and r['backend'] != 'unknown'})
    return successes, throttled, unavailable, served_backends


tests = ApimTesting('Inference Failover Scenario Tests', sample_folder, nb_helper.deployment)

session = http_requests.Session()
session.verify = not allow_insecure_tls
if request_headers:
    session.headers.update(request_headers)
session.headers.update({'Content-Type': 'application/json'})

try:
    # 1/5: Baseline warm path - both routes, small payloads against fresh priority-1 backends.
    scenario1_gpt_5_1 = run_inference_scenario(
        session, gpt_5_1_route, api_keys['inference-gpt-5-1'], small_payload, 6, gpt_5_1_backend_index,
        '1/5: Baseline warm path - gpt-5.1',
    )
    scenario1_gpt_4_1_mini = run_inference_scenario(
        session, gpt_4_1_mini_route, api_keys['inference-gpt-4-1-mini'], small_payload, 6, gpt_4_1_mini_backend_index,
        '1/5: Baseline warm path - gpt-4.1-mini',
    )
    tests.verify(len(scenario1_gpt_5_1), 6)
    tests.verify(len(scenario1_gpt_4_1_mini), 6)
    tests.verify(scenario1_gpt_5_1[0]['status_code'], 200, label = 'gpt-5.1 baseline first request succeeded')
    tests.verify(scenario1_gpt_4_1_mini[0]['status_code'], 200, label = 'gpt-4.1-mini baseline first request succeeded')

    # 2/5: Sustained pressure on gpt-5.1 - no spacing, drive low-TPM exhaustion and failover.
    time.sleep(2)
    scenario2_gpt_5_1 = run_inference_scenario(
        session, gpt_5_1_route, api_keys['inference-gpt-5-1'], pressure_payload, 45, gpt_5_1_backend_index,
        '2/5: Sustained pressure - gpt-5.1 (no spacing)',
    )
    s2_success, s2_throttled, s2_unavailable, _ = summarize_scenario(scenario2_gpt_5_1)
    tests.verify(len(scenario2_gpt_5_1), 45)
    tests.verify(s2_success > 0, True, label = 'gpt-5.1 sustained pressure still served some requests')
    tests.verify(
        (s2_throttled + s2_unavailable) > 0,
        True,
        label = 'gpt-5.1 sustained pressure produced throttling or failover exhaustion',
    )

    # 3/5: Sustained pressure on gpt-4.1-mini - the independent pool exhausts on its own.
    time.sleep(2)
    scenario3_gpt_4_1_mini = run_inference_scenario(
        session, gpt_4_1_mini_route, api_keys['inference-gpt-4-1-mini'], pressure_payload, 30, gpt_4_1_mini_backend_index,
        '3/5: Sustained pressure - gpt-4.1-mini (no spacing)',
    )
    s3_success, s3_throttled, s3_unavailable, _ = summarize_scenario(scenario3_gpt_4_1_mini)
    tests.verify(len(scenario3_gpt_4_1_mini), 30)
    tests.verify(s3_success > 0, True, label = 'gpt-4.1-mini sustained pressure still served some requests')
    tests.verify(
        (s3_throttled + s3_unavailable) > 0,
        True,
        label = 'gpt-4.1-mini sustained pressure produced throttling or failover exhaustion',
    )

    # 4/5: Paced recovery on gpt-5.1 - inter-request spacing lets circuit breakers recover.
    time.sleep(2)
    scenario4_gpt_5_1 = run_inference_scenario(
        session, gpt_5_1_route, api_keys['inference-gpt-5-1'], pressure_payload, 30, gpt_5_1_backend_index,
        '4/5: Paced recovery - gpt-5.1 (1.5s spacing)', sleep_ms = 1500,
    )
    s4_success, _, _, _ = summarize_scenario(scenario4_gpt_5_1)
    tests.verify(len(scenario4_gpt_5_1), 30)
    tests.verify(s4_success > 0, True, label = 'gpt-5.1 paced recovery served requests across the window')

    # 5/5: Terminal exhaustion on gpt-5.1 - a hard burst drains every tier to the generic 503.
    time.sleep(2)
    scenario5_gpt_5_1 = run_inference_scenario(
        session, gpt_5_1_route, api_keys['inference-gpt-5-1'], pressure_payload, 35, gpt_5_1_backend_index,
        '5/5: Terminal exhaustion - gpt-5.1 (hard burst)',
    )
    s5_success, s5_throttled, s5_unavailable, _ = summarize_scenario(scenario5_gpt_5_1)
    tests.verify(len(scenario5_gpt_5_1), 35)
    tests.verify(
        (s5_throttled + s5_unavailable) > 0,
        True,
        label = 'gpt-5.1 terminal burst exhausted the pool (429/503 observed)',
    )
finally:
    session.close()

print_message('Scenario success / throttled / unavailable summary', blank_above = True)
for scenario_name, scenario_results in [
    ('1 baseline gpt-5.1', scenario1_gpt_5_1),
    ('1 baseline gpt-4.1-mini', scenario1_gpt_4_1_mini),
    ('2 sustained gpt-5.1', scenario2_gpt_5_1),
    ('3 sustained gpt-4.1-mini', scenario3_gpt_4_1_mini),
    ('4 paced gpt-5.1', scenario4_gpt_5_1),
    ('5 terminal gpt-5.1', scenario5_gpt_5_1),
]:
    n_ok, n_throttled, n_unavailable, served = summarize_scenario(scenario_results)
    print_info(f'Scenario {scenario_name}: {n_ok} ok, {n_throttled} throttled, {n_unavailable} unavailable, backends={served}')

tests.print_summary()
if tests.tests_failed:
    raise SystemExit(1)
print_ok('All inference failover scenarios completed')


## 📊 Scenario Charts

Plot the per-run response time for each scenario. Bars are colored by the backend that served the request, derived from the `X-Backend-Id` response header that the inference policy emits.

**Reading the charts**

- **Backend index 0** is the priority-1 backend; higher indices are lower-priority failover tiers in pool order.
- A run colored red is a non-`200` response (`429` throttle or terminal `503`) and shows pressure that the pool could not absorb.
- The first request in a cold scenario is often slower (TLS warm-up plus first-token latency); the mean line ignores the slowest 5% so it reflects steady-state behaviour.
- As pressure builds you should see the colors walk from index 0 upward as the priority-1 backend trips and traffic fails over to lower tiers.

**Backend index legend**

| Index | gpt-5.1 pool | gpt-4.1-mini pool |
| --- | --- | --- |
| 0 | a-gpt-5-1 (P1) | c-gpt-4-1-mini (P1) |
| 1 | b-gpt-5-1 (P2) | d-gpt-4-1-mini (P2) |
| 2 | d-gpt-5-1 (P3) | f-gpt-4-1-mini (P3) |
| 3 | e-gpt-5-1 (P4) | h-gpt-4-1-mini (P4) |
| 4 | g-gpt-5-1 (P4) | - |

> If every bar is the same color and labeled index 99, the gateway did not rewrite the request URL to the resolved pool member (the `X-Backend-Id` header returned `unknown`). The scenarios and the server-side distribution below remain valid; only the per-run backend coloring is unavailable.


In [ ]:
import charts
from console import print_error

if 'scenario5_gpt_5_1' not in locals():
    print_error('Please run the scenario cell first')
    raise SystemExit(1)

scenario_charts = [
    (scenario1_gpt_5_1, 'Scenario 1 - Baseline warm path (gpt-5.1)',
     'Small requests on a fresh pool stay on the priority-1 backend (index 0).'),
    (scenario1_gpt_4_1_mini, 'Scenario 1 - Baseline warm path (gpt-4.1-mini)',
     'Small requests on a fresh pool stay on the priority-1 backend (index 0).'),
    (scenario2_gpt_5_1, 'Scenario 2 - Sustained pressure (gpt-5.1, no spacing)',
     'Low-TPM exhaustion forces throttling and failover across tiers; red bars are non-200 responses.'),
    (scenario3_gpt_4_1_mini, 'Scenario 3 - Sustained pressure (gpt-4.1-mini, no spacing)',
     'The gpt-4.1-mini pool exhausts independently of gpt-5.1.'),
    (scenario4_gpt_5_1, 'Scenario 4 - Paced recovery (gpt-5.1, 1.5s spacing)',
     'Inter-request spacing lets circuit breakers recover, lifting the success rate versus scenario 2.'),
    (scenario5_gpt_5_1, 'Scenario 5 - Terminal exhaustion (gpt-5.1, hard burst)',
     'A hard burst drains every tier; terminal 503s appear once all backends are tripped.'),
]

for scenario_results, chart_title, chart_note in scenario_charts:
    charts.BarChart(
        api_results = scenario_results,
        title = chart_title,
        x_label = 'Run #',
        y_label = 'Response Time (ms)',
        fig_text = chart_note,
    ).plot()


## [Verify] Wait For AI Gateway Telemetry

Poll the existing Log Analytics workspace for gateway rows and token-bearing LLM diagnostic rows created by the traffic block. Ingestion normally takes several minutes.

In [ ]:
from pathlib import Path

from console import print_error, print_ok, print_warning

if 'log_analytics_id' not in locals():
    print_error('Please run the deployment and traffic cells first')
    raise SystemExit(1)

api_ids_binding = "let apiIds = dynamic(['inference-gpt-5-1', 'inference-gpt-4-1-mini']);\n"
time_binding = 'let timeWindow = 30m;\n'
verify_path = utils.determine_policy_path('verify-llm-ingestion.kql', sample_folder)
verify_template = Path(verify_path).read_text(encoding='utf-8')
verification_query = time_binding + api_ids_binding + verify_template
telemetry_found, telemetry_result, telemetry_rows = nb_helper.wait_for_kql(
    log_analytics_id,
    verification_query,
    api_path='/api/query',
    api_version='2020-08-01',
    progress_label='AI telemetry rows',
)

if telemetry_found:
    request_rows, successful_requests, unavailable_responses, token_rows, total_tokens = telemetry_rows[0]
    print_ok(f'Telemetry ready: {request_rows} requests, {token_rows} token-bearing requests, {total_tokens} tokens')
    if unavailable_responses:
        print_ok(f'Observed {unavailable_responses} terminal 503 responses after fallback exhaustion')
elif telemetry_result is not None and telemetry_result.success:
    print_warning('Token-bearing telemetry has not arrived yet; wait a few minutes and rerun this cell')

## [Verify] Plot Backend Distribution And Token Volume

Query the same AOAI-only data surfaced in the deployed workbook and render local graphs for route distribution and token consumption.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from console import print_info, print_warning

distribution_path = utils.determine_policy_path('backend-distribution.kql', sample_folder)
token_path = utils.determine_policy_path('token-throughput.kql', sample_folder)
distribution_template = Path(distribution_path).read_text(encoding='utf-8')
token_template = Path(token_path).read_text(encoding='utf-8')
bindings = "let timeWindow = 1h;\nlet apiIds = dynamic(['inference-gpt-5-1', 'inference-gpt-4-1-mini']);\n"
distribution_found, _, distribution_rows = nb_helper.wait_for_kql(
    log_analytics_id,
    bindings + distribution_template,
    schedule=[0],
    progress_label='distribution rows',
)
tokens_found, _, token_rows = nb_helper.wait_for_kql(
    log_analytics_id,
    bindings + token_template,
    schedule=[0],
    progress_label='token rows',
)

if distribution_found:
    distribution_frame = pd.DataFrame(
        distribution_rows,
        columns=['API', 'Backend', 'Requests', 'Successes', 'Throttled', 'Failures', 'AverageBackendMs', 'SuccessRate'],
    )
    display(distribution_frame)
    distribution_frame.pivot(index='Backend', columns='API', values='Requests').fillna(0).plot(
        kind='bar',
        figsize=(12, 5),
        title='Gateway-recorded request distribution by backend',
    )
    plt.xlabel('Backend')
    plt.ylabel('Requests')
    plt.tight_layout()
    plt.show()
else:
    print_warning('No gateway distribution rows returned yet')

if tokens_found:
    token_frame = pd.DataFrame(
        token_rows,
        columns=['API', 'Backend', 'Model', 'Requests', 'PromptTokens', 'CompletionTokens', 'TotalTokens'],
    )
    display(token_frame)
    token_frame.groupby('API')['TotalTokens'].sum().plot(
        kind='bar',
        figsize=(8, 4),
        title='LLM token volume by inference API',
    )
    plt.xlabel('Inference API')
    plt.ylabel('Total tokens')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print_info('No token rows returned yet; successful inference rows may still be ingesting')

## Open Azure Views

Print links to the deployed workbook, Log Analytics workspace, and APIM instance for deeper exploration after the notebook graphs complete.

In [ ]:
from console import print_ok, print_val

portal_base = f'https://portal.azure.com/#@{tenant_id}/resource'
apim_id = (
    f'/subscriptions/{subscription_id}/resourceGroups/{rg_name}'
    f'/providers/Microsoft.ApiManagement/service/{apim_name}'
)
print_val('Workbook', f'{portal_base}{workbook_id}/workbook')
print_val('Log Analytics', f'{portal_base}{log_analytics_id}/logs')
print_val('API Management', f'{portal_base}{apim_id}/overview')
print_ok('All done!')